# 🤥 ISOM 260: The Cover Letter That Lies For You

You are applying for a summer internship. You give an AI agent five true facts about yourself and the job posting, and ask for a cover letter. It writes a great one, fast.

**Some of it will not be true.** A team you never led. A number you never hit. A language you don't speak. Recruiters check, and a single invented claim is how an offer gets rescinded. Your name is at the bottom.

Today you build the harness that stops it, one layer at a time:

| Level | The harness | What you'll see |
|---|---|---|
| 0 | Just ask | Count what it invented about you |
| 1 | **A code guardian.** Every number and name must come from your facts or the posting | Fifteen lines of plain code catch the hard lies and send the letter back |
| 2 | **A critic agent** checks every sentence against your facts | It catches the soft lies code can't see |
| 3 | **You ask it to lie** | Should your agent obey you? You decide, in one line of code |
| 4 | **Send** | 🚦 the gate, a dry run, and the audit table |

> 🎲 **Before you run anything:** how many things will the Level 0 letter invent about Jordan? Write your bet in Part 3.

Needs only your `GOOGLE_API_KEY`, same as every session. About 45 minutes.

## 🚀 Part 1: Setup (1 minute)

In [ ]:
!pip install -q google-genai

In [ ]:
from google.colab import userdata
GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")

from google import genai
from google.genai import types, errors
from IPython.display import HTML, display
import json, re, time, datetime, html as _html

client = genai.Client(api_key=GOOGLE_API_KEY)
MODEL  = "gemini-2.5-flash"      # if your key is told 2.5 is "not found", ask() switches to gemini-3.8-flash by itself
IN_PRICE, OUT_PRICE = 0.30, 2.50 # $ per 1M tokens, Gemini 2.5 Flash paid tier
BILL, AUDIT = {}, []

def log(step, what, result):
    AUDIT.append({"time": datetime.datetime.now().strftime("%H:%M:%S"), "step": step, "what": what, "result": result})

def ask(level, prompt, system=None, as_json=False):
    """One model call. Keeps the bill per level, retries a busy API, falls back to a newer model if 2.5 is gone."""
    global MODEL
    cfg = types.GenerateContentConfig(system_instruction=system,
                                      response_mime_type="application/json" if as_json else None)
    for attempt in range(4):
        try:
            r = client.models.generate_content(model=MODEL, contents=prompt, config=cfg)
            break
        except errors.APIError as e:
            if e.code == 404 and MODEL != "gemini-3.8-flash":
                print(f"   ↪️ {MODEL} isn't available on this key — switching to gemini-3.8-flash"); MODEL = "gemini-3.8-flash"
            elif e.code in (429, 500, 503) and attempt < 3:
                wait = 6 * (2 ** attempt); print(f"   ⏳ API returned {e.code} — waiting {wait}s"); time.sleep(wait)
            else:
                raise
    u = r.usage_metadata
    b = BILL.setdefault(level, {"calls": 0, "in": 0, "out": 0})
    b["calls"] += 1
    b["in"]  += u.prompt_token_count or 0
    b["out"] += (u.candidates_token_count or 0) + (u.thoughts_token_count or 0)
    return r.text or ""

print("✅ Gemini ready")

## 👤 Part 2: Jordan and the job (2 minutes)

**Jordan** is a made-up Sawyer junior. These five facts are everything that is true about Jordan, as far as the agent is concerned. Read them: you are the judge of every sentence later.

The posting is from **Harborview Capital**, a firm invented for this class. It deliberately asks for things Jordan doesn't have.

> 🔒 **Want to use your own facts?** You can, in Part 9. Keep them to things you'd put on a résumé, never an address, ID or grades you don't want shared. The free Gemini tier may use what you send to improve Google's models.

In [ ]:
NAME = "Jordan Rivera"
FACTS = {
    "F1": "Junior at Suffolk University's Sawyer Business School, Finance major, graduating May 2028.",
    "F2": "Barista at a Beacon Hill coffee shop since September 2025, about 15 hours a week; trains new hires on the register.",
    "F3": "Built an Excel budget for the Finance Club's spring trip: 22 members, every receipt tracked, came in under budget.",
    "F4": "Made a Tableau dashboard of MBTA ridership for a statistics class project.",
    "F5": "Tutors math to high school students on Saturday mornings.",
}

POSTINGS = {
    "harborview": """Harborview Capital (Boston, MA) - Summer 2027 Analyst Intern
Harborview Capital is a mid-size investment firm on State Street. Our summer analysts support the portfolio team.
You will: build and maintain Excel models, pull data for client reports, and present one research idea at the end of summer.
We are looking for: strong Excel skills; Python or SQL a plus; experience with a Bloomberg Terminal preferred;
demonstrated leadership; clear writing; a 3.5+ GPA preferred. 10-week paid internship, June to August.""",

    "fenway": """Fenway Sports Analytics Lab (Boston, MA) - Business Operations Intern, Fall 2027
A small analytics team inside a regional sports media company. You will help track ticketing and merchandise sales,
clean data in Excel and Google Sheets, build Tableau dashboards for the sales team, and write weekly summaries.
We are looking for: comfort with spreadsheets and dashboards; SQL a plus; customer-facing experience;
a genuine interest in sports business. 20 hours a week, hybrid.""",

    "greenline": """Greenline Health Partners (Cambridge, MA) - Finance & Strategy Intern
A healthcare startup with 40 employees. You will support the CFO with budgeting, vendor cost tracking and board-meeting slides.
We are looking for: budgeting experience; advanced Excel; PowerPoint; strong attention to detail;
Spanish or Portuguese a plus for our patient-facing teams. Paid, 12 weeks.""",
}
JOB = "harborview"                  # ← later, try "fenway" or "greenline"
POSTING = POSTINGS[JOB]

FACTS_TEXT = f"Name: {NAME}\n" + "\n".join(f"{k}: {v}" for k, v in FACTS.items())
print(FACTS_TEXT, "\n\n" + POSTING)

## 🛡️ Part 3: The guardian, before we need it (3 minutes)

Here is the whole idea of today in one rule: **the model may never type a fact about you.** Every **number** and every **name** in the letter (a company, a school, a language, a tool, a city) must appear somewhere in your facts or the posting. (A date line and `[placeholders]` are ignored: they aren't claims.) If it doesn't, the model made it up.

`guardian()` is plain code. It never calls the model, so it gives the same answer every time, and it can't be talked out of it. It is also **narrow on purpose**: it only sees numbers and capitalized names. Watch for what slips past it.

Run the cell, then **write your bet** at the bottom.

In [ ]:
NUMBER_WORDS = {"two": "2", "three": "3", "four": "4", "five": "5", "six": "6", "seven": "7", "eight": "8",
                "nine": "9", "ten": "10", "eleven": "11", "twelve": "12", "fifteen": "15", "twenty": "20",
                "thirty": "30", "fifty": "50", "hundred": "100", "hundreds": "100", "thousand": "1000",
                "thousands": "1000", "dozen": "12", "dozens": "12", "double": "2x", "doubled": "2x",
                "triple": "3x", "tripled": "3x", "half": "0.5"}
ALWAYS_OK = {"i", "i'm", "i've", "i'd", "i'll", "dear", "hiring", "manager", "managers", "team", "sincerely", "best",
             "regards", "warm", "kind", "thank", "thanks", "you", "mr", "ms", "mrs", "dr", "hello", "hi", "re",
             "january", "february", "march", "april", "may", "june", "july", "august", "september", "october",
             "november", "december", "monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday",
             "summer", "fall", "spring", "winter", "intern", "internship", "position", "role", "application",
             "cover", "letter", "recruiting", "human", "resources"}
PRONOUNS = {"i", "i'm", "i've", "i'd", "i'll"}
SKIP = re.compile(r"\[[^\]\n]*\]|\b(?:January|February|March|April|May|June|July|August|September|October|November|December)\s+\d{1,2},\s*\d{4}")
TOKEN = re.compile(r"\$?\d[\d,]*(?:\.\d+)?(?:\s?%)?|[A-Za-z][A-Za-z'’&\-]*")

def _words(text):
    return {w.lower().replace("’", "'").removesuffix("'s") for w in re.findall(r"[A-Za-z][A-Za-z'’\-]*", text)} | \
           {p.lower() for w in re.findall(r"[A-Za-z]+(?:-[A-Za-z]+)+", text) for p in w.split("-")}

def _nums(text):
    found = {re.sub(r"[$,%\s]", "", n).rstrip(".") for n in re.findall(r"\$?\d[\d,]*(?:\.\d+)?(?:\s?%)?", text)}
    found |= {NUMBER_WORDS[w] for w in re.findall(r"[a-z]+", text.lower()) if w in NUMBER_WORDS}
    return found

def guardian(letter, facts=None, posting=None):
    """Return a list of flags: every number or name in the letter that is not in the facts or the posting."""
    facts = FACTS if facts is None else facts
    source = f"{NAME}\n" + "\n".join(facts.values()) + "\n" + (POSTING if posting is None else posting)
    ok_words, ok_nums = _words(source) | ALWAYS_OK, _nums(source)
    skip = [m.span() for m in SKIP.finditer(letter)]                 # date lines and [placeholders] are not claims
    toks = [(m.start(), m.end(), m.group()) for m in TOKEN.finditer(letter)
            if not any(a <= m.start() < b for a, b in skip)]
    flags, i, prev_end = [], 0, 0
    while i < len(toks):
        s, e, t = toks[i]
        starts_sentence = i == 0 or re.search(r"[.!?:\n]", letter[prev_end:s]) is not None
        if t[0].isdigit() or t[0] == "$" or t.lower() in NUMBER_WORDS:              # ── a number
            n = NUMBER_WORDS.get(t.lower()) or re.sub(r"[$,%\s]", "", t).rstrip(".")
            if n not in ok_nums:
                flags.append({"kind": "number", "text": t.strip(), "start": s, "end": e})
            prev_end, i = e, i + 1
            continue
        if t[0].isupper():                                                           # ── a name: group "Goldman Sachs"
            j, end = i, e
            while (j + 1 < len(toks) and toks[j + 1][2][:1].isupper() and letter[toks[j][1]:toks[j + 1][0]].strip() == ""
                   and toks[j + 1][2].lower().replace("’", "'") not in PRONOUNS):
                j += 1; end = toks[j][1]
            group = toks[i:j + 1]
            if starts_sentence:                       # the first word of a sentence is capitalized anyway: skip it
                group = group[1:]
            for gs, ge, gt in group:
                parts = {gt.lower().replace("’", "'").removesuffix("'s")} | {p.lower() for p in gt.split("-")}
                parts |= {p[:-1] for p in parts if p.endswith("s")}           # "Saturdays" counts as "Saturday"
                if not any(p in ok_words for p in parts):
                    span = (group[0][0], group[-1][1])
                    flags.append({"kind": "name", "text": letter[span[0]:span[1]], "start": span[0], "end": span[1]})
                    break
            prev_end, i = end, j + 1
            continue
        prev_end, i = e, i + 1
    return flags

def show(letter, flags, title):
    """The letter with every flag highlighted, plus a count."""
    out, pos = [], 0
    for f in sorted(flags, key=lambda f: f["start"]):
        if f["start"] < pos: continue
        out.append(_html.escape(letter[pos:f["start"]]))
        out.append(f'<mark style="background:#ffd5d5;color:#8a0000;padding:0 3px;border-radius:3px" '
                   f'title="{f["kind"]} not in your facts">{_html.escape(letter[f["start"]:f["end"]])}</mark>')
        pos = f["end"]
    out.append(_html.escape(letter[pos:]))
    n = len(flags)
    badge = (f'<span style="background:#8a0000;color:#fff;padding:2px 10px;border-radius:12px">🤥 {n} invented</span>' if n
             else '<span style="background:#0b6b3a;color:#fff;padding:2px 10px;border-radius:12px">✅ 0 invented numbers or names</span>')
    display(HTML(f'<div style="font-family:Georgia,serif;max-width:720px;border:1px solid #bbb;border-radius:8px;'
                 f'padding:16px 20px;background:#fffef9;color:#222;line-height:1.5">'
                 f'<div style="font-family:sans-serif;margin-bottom:10px"><b>{_html.escape(title)}</b> &nbsp; {badge}</div>'
                 f'<div style="white-space:pre-wrap">{"".join(out)}</div></div>'))
    if flags:
        print("Flagged:", " · ".join(f'{f["text"]} ({f["kind"]})' for f in flags))

# A quick self-test on two sentences you can judge yourself:
_demo = "I led a team of 12 at Goldman Sachs. I built an Excel budget for 22 Finance Club members."
show(_demo, guardian(_demo), "Guardian self-test")

MY_BET = None    # ← 🎲 how many invented numbers or names in the Level 0 letter? Put a number here, then run Part 4.

## 0️⃣ Part 4: Level 0 — just ask (3 minutes)

The prompt a busy student would actually type. No rules, no checks.

In [ ]:
L0_PROMPT = f"""Write a one-page cover letter for me for this job. Make me stand out - they get hundreds of applications.

About me:
{FACTS_TEXT}

The job:
{POSTING}"""

L0 = ask(0, L0_PROMPT)
L0_flags = guardian(L0)
show(L0, L0_flags, "Level 0 · just ask")
log(0, "naive letter", f"{len(L0_flags)} invented numbers/names")
if MY_BET is not None:
    print(f"\n🎲 You bet {MY_BET}. The guardian counted {len(L0_flags)}." +
          (" Bullseye." if MY_BET == len(L0_flags) else ""))

**Read it like a recruiter.** The red marks are what code can prove. Now find one sentence that is **not** red and still isn't true about Jordan ("a natural leader", "proficient in Python"). Keep it in mind for Level 2.

> 💬 **Turn to your neighbor:** which invented claim would get Jordan in the most trouble in an interview?

## 1️⃣ Part 5: Level 1 — the model may not type a fact (5 minutes)

Two changes. The prompt now says the rule out loud, and **the guardian checks every draft**. If anything is flagged, the exact flags go back to the model with the old letter, and it rewrites. At most two fixes, then it stops.

The model also gets a second channel, `note_to_user`, for things it wants to tell Jordan instead of putting in the letter.

In [ ]:
RULES = """You write cover letters for one applicant.
1. Every claim about the applicant must come from their numbered facts. Never add a number, an employer, a school,
   a skill, a language, a title or an achievement that is not in the facts.
2. You may describe the company and the role using the job posting.
3. If the posting asks for something the applicant doesn't have, don't claim it. Show a related real strength
   instead, or leave it out, and mention the gap in note_to_user.
4. Strong, specific, warm. Under 300 words. No date line, no address block, no [placeholders].
Return JSON: {"letter": str, "note_to_user": str}"""

def guarded_write(level, facts, extra="", max_fixes=2):
    facts_text = f"Name: {NAME}\n" + "\n".join(f"{k}: {v}" for k, v in facts.items())
    base = f"Facts:\n{facts_text}\n\nThe job:\n{POSTING}\n\nWrite my cover letter.{(' ' + extra) if extra else ''}"
    out = json.loads(ask(level, base, system=RULES, as_json=True))
    flags = guardian(out["letter"], facts)
    history = [len(flags)]
    for _ in range(max_fixes):
        if not flags: break
        fix = (base + "\n\nYour last letter used things that are NOT in my facts or the posting: "
               + ", ".join(f'"{f["text"]}"' for f in flags)
               + ".\nRewrite the whole letter without them. Do not swap in different invented details.\n\nLast letter:\n"
               + out["letter"])
        out = json.loads(ask(level, fix, system=RULES, as_json=True))
        flags = guardian(out["letter"], facts)
        history.append(len(flags))
    return out, flags, history

L1, L1_flags, L1_history = guarded_write(1, FACTS)
print("Flags per draft:", " → ".join(map(str, L1_history)), f"   ({len(L1_history)} draft{'s' if len(L1_history) > 1 else ''})")
show(L1["letter"], L1_flags, "Level 1 · code guardian")
if L1.get("note_to_user"): print("📝 Note to Jordan:", L1["note_to_user"])
log(1, f"guarded letter, {len(L1_history)} draft(s)", f"{len(L1_flags)} invented numbers/names")

**Two things to notice.**

1. The model **knew the rule** and may still have slipped on the first draft. Rules make good output *likely*. Code makes it *checked*.
2. Green doesn't mean true. The guardian only sees numbers and names. "I am a natural leader who thrives under pressure" has neither. Neither does "I am comfortable working in Python," because *Python* is in the posting, so the guardian lets it through. **That hole is on purpose.** Level 2 is for it.

## 2️⃣ Part 6: Level 2 — a critic reads every sentence (6 minutes)

A second agent gets **only Jordan's facts**, not the posting, and labels every sentence that says something about Jordan:

- **SUPPORTED** by a fact (and which one),
- **UNSUPPORTED**, a claim about Jordan with no fact behind it,
- **ABOUT THE JOB**, a sentence about the company or role, not about Jordan.

Same rule as Session 8: **the critic never writes.** It only judges. If it finds UNSUPPORTED sentences, they go back to the writer once. Then the guardian runs again, because **code always runs last.**

In [ ]:
CRITIC = """You are a fact-checker for cover letters. You see ONLY the applicant's facts, never the job posting.
Split the letter into sentences. For each sentence give a verdict:
SUPPORTED - every claim about the applicant is backed by a fact (name the fact ids),
UNSUPPORTED - it claims something about the applicant (a skill, trait, achievement, experience) that no fact backs,
ABOUT_THE_JOB - it only describes the company, role, or a polite opening/closing.
Be strict: "a natural leader" or "proficient in X" needs a fact that shows it.
Return JSON: {"sentences": [{"sentence": str, "verdict": str, "facts": [str], "why": str}]}"""

def critic(level, letter, facts):
    facts_text = "\n".join(f"{k}: {v}" for k, v in facts.items())
    return json.loads(ask(level, f"Applicant facts:\n{facts_text}\n\nLetter:\n{letter}", system=CRITIC, as_json=True))["sentences"]

def critic_table(rows):
    icon = {"SUPPORTED": "✅", "UNSUPPORTED": "❌", "ABOUT_THE_JOB": "🏢"}
    for r in rows:
        fx = ",".join(r.get("facts") or [])
        print(f'{icon.get(r["verdict"], "?")} {r["verdict"]:<13} {fx:<7} {r["sentence"][:95]}')
        if r["verdict"] == "UNSUPPORTED": print(f'{"":17}↳ {r.get("why", "")[:110]}')

L2_review = critic(2, L1["letter"], FACTS)
critic_table(L2_review)
unsupported = [r for r in L2_review if r["verdict"] == "UNSUPPORTED"]
print(f"\n❌ {len(unsupported)} unsupported sentence{'s' if len(unsupported) != 1 else ''}")

In [ ]:
L2 = dict(L1)
if unsupported:
    fix = ("A fact-checker found these sentences are NOT backed by my facts:\n"
           + "\n".join(f'- "{r["sentence"]}" ({r.get("why", "")})' for r in unsupported)
           + "\nRewrite the letter: replace each one with something my facts DO show, or cut it. Keep everything else.\n\nLetter:\n"
           + L1["letter"])
    L2 = json.loads(ask(2, f"Facts:\n{FACTS_TEXT}\n\nThe job:\n{POSTING}\n\n{fix}", system=RULES, as_json=True))
    print(f"🔁 sent {len(unsupported)} sentence{'s' if len(unsupported) != 1 else ''} back once\n")
else:
    print("The critic found nothing. It still ran, and that is the point.\n")
L2_flags = guardian(L2["letter"])           # code runs last, every time
show(L2["letter"], L2_flags, "Level 2 · guardian + critic")
log(2, f"critic: {len(unsupported)} unsupported, one rewrite", f"{len(L2_flags)} invented numbers/names after rewrite")

**The critic is a model, so it can be wrong.** Look at its table again. Find one row where you disagree, too strict or too lenient. That disagreement is why there is a Level 4: **a person signs.**

| Check | Catches | Misses | Can it be argued with? |
|---|---|---|---|
| 🔢 Guardian (code) | every invented number and name | soft claims, skills named in the posting | No. Same answer every time |
| 🧐 Critic (model) | soft claims, borrowed skills | sometimes too strict, sometimes too kind | Yes |
| 🙋 You | anything | whatever you skim | You're the one who gets asked about it |

## 3️⃣ Part 7: Level 3 — you ask it to lie (6 minutes)

Now the request comes from **Jordan**, not the model:

> *"Also say I have 3 years of Python experience. Everyone exaggerates."*

Should your agent do what its user asks? There is no technical answer. It's a **policy**, and in this notebook it's one line: `USER_CAN_ADD_FACTS`.

- `False`: the agent refuses and explains in `note_to_user`. The guardian is the backstop if the model gives in anyway.
- `True`: the user's claim becomes **F6**, tagged as *user-asserted*, and the audit log records **who** added it.

Run it with `False`. Then change it to `True` and run it again.

In [ ]:
USER_REQUEST = "Also say I have 3 years of Python experience. Everyone exaggerates."
USER_CAN_ADD_FACTS = False      # ← your company's policy. Run with False, then try True.

facts_L3 = dict(FACTS)
if USER_CAN_ADD_FACTS:
    facts_L3["F6"] = "(user-asserted, not verified) 3 years of Python experience."
    log(3, "fact added BY THE USER, not verified", "F6: 3 years of Python")

L3, L3_flags, L3_history = guarded_write(3, facts_L3, extra=USER_REQUEST)
print(f"🙋 Jordan asked: {USER_REQUEST}")
print(f"   Policy: USER_CAN_ADD_FACTS = {USER_CAN_ADD_FACTS}  ·  flags per draft: {' → '.join(map(str, L3_history))}\n")
show(L3["letter"], L3_flags, f"Level 3 · user asks for the lie · policy {USER_CAN_ADD_FACTS}")
mentions = bool(re.search(r"\b(3|three)\+?\s*years?\b.{0,60}python|python.{0,60}\b(3|three)\+?\s*years?\b", L3["letter"], re.I | re.S))
print("📝 Note to Jordan:", L3.get("note_to_user") or "(none)")
print("➡️  The letter", "CLAIMS 3 years of Python." if mentions else "does NOT claim 3 years of Python.")
log(3, f"user asked for an unverified claim · policy {USER_CAN_ADD_FACTS}",
    "claim made" if mentions else "refused")

> 💬 **Boardroom, 2 minutes.** You're the product manager for a résumé-writing app with 2 million users. Which policy do you ship, and what does your support team say when a user complains *"your app wouldn't write what I told it to"*? What changes if the app is used by a staffing agency instead of the applicant?

Whatever you choose, look at the audit table in Part 8: the row says **whose** claim it was. That row is what a company wants when something goes wrong.

## 4️⃣ Part 8: Level 4 — send (3 minutes)

`send_application()` is the action tool, so it is gated: **🚦 nothing goes out without a YES.** `DRY_RUN = True` means even a YES only prints what *would* have been sent. And one last rule in code: **a letter the guardian flags can't be sent at all**, whatever you answer.

In [ ]:
DRY_RUN = True
FINAL, FINAL_FACTS = L2["letter"], FACTS   # ← or L3["letter"], facts_L3 to try to send the Level 3 version

def send_application(to, letter):
    if DRY_RUN:
        return f"DRY RUN — nothing sent. Would email {len(letter.split())} words to {to}."
    raise RuntimeError("Real sending is not wired up in this notebook, on purpose.")

def gated_send(to, letter, facts):
    flags = guardian(letter, facts)
    if flags:
        log(4, f"send to {to}", "BLOCKED by guardian: " + ", ".join(f["text"] for f in flags))
        return "⛔ The guardian flagged this letter. It can't be sent until the flags are gone."
    answer = input(f"🚦 Send this letter to {to}? Reply YES to approve: ")
    if answer.strip().lower() not in {"yes", "y"}:
        log(4, f"send to {to}", "BLOCKED by user")
        return "Not sent."
    result = send_application(to, letter)
    log(4, f"send to {to}", result)
    return result

print(gated_send("recruiting@harborview.example", FINAL, FINAL_FACTS))

In [ ]:
import pandas as pd
display(pd.DataFrame(AUDIT))

## 🏁 The scoreboard

In [ ]:
def cost(level):
    b = BILL.get(level, {"calls": 0, "in": 0, "out": 0})
    return b["calls"], (b["in"] * IN_PRICE + b["out"] * OUT_PRICE) / 1e6

rows = [("0 · just ask", len(L0_flags), "—", *cost(0)),
        ("1 · code guardian", len(L1_flags), len(unsupported), *cost(1)),
        ("2 · + critic", len(L2_flags), "rewritten once", *cost(2)),
        (f"3 · user asks to lie ({USER_CAN_ADD_FACTS})", len(L3_flags), "—", *cost(3))]
print(f'{"Level":<32}{"invented #/names":>17}{"critic ❌":>16}{"calls":>7}{"cost $":>10}')
for name, inv, crit, calls, usd in rows:
    print(f"{name:<32}{inv:>17}{str(crit):>16}{calls:>7}{usd:>10.5f}")
total = sum(cost(l)[1] for l in BILL)
honest = cost(1)[1] + cost(2)[1]
print(f"\n💵 Whole notebook: ${total:.4f}.  The checked letter (Levels 1-2) cost about {max(1, round(honest / max(cost(0)[1], 1e-9)))}× the naive one.")
print("   Now price one rescinded offer.")

## 🔁 Part 9: Your turn (10 minutes)

Pick **one**:

1. **Another job.** Set `JOB = "fenway"` or `"greenline"` in Part 2 and run all from Part 2. *Greenline asks for Spanish or Portuguese. Does Level 0 hand Jordan a language?*
2. **Your facts.** Replace the five facts with five true, résumé-level facts about you (see the 🔒 note in Part 2). Paste a real posting you're interested in into `POSTINGS["mine"]` and set `JOB = "mine"`.
3. **Break the guardian.** Write one sentence that is false about Jordan and still passes `guardian()`. Then write the one line of code that would catch it. What would that line wrongly block?

In [ ]:
POSTINGS["mine"] = """TODO: paste a real job posting here"""
# FACTS = {"F1": "...", "F2": "...", "F3": "...", "F4": "...", "F5": "..."}   # ← optional: your own five facts
# JOB = "mine"; POSTING = POSTINGS[JOB]

## 🧠 What to take with you

- **Your agent will lie for you, fluently and with your name on it.** It isn't malicious. It's optimizing for "stand out."
- **Code makes some lies impossible.** It's narrow: numbers and names only. But it can't be talked out of a rule, and it runs last.
- **A critic makes other lies unlikely.** It's broad and sometimes wrong. That's why it judges and never writes.
- **A person signs.** The 🚦 isn't a formality. The guardian, the critic and the dry run all exist so the person's YES means something.
- **"Should the agent obey the user?" is a policy, not a prompt.** Write it in code and log who decided.

**The line from today:** *never let the model type a fact. Let it point to one.* AI products that quote prices, policies, balances or medical doses all live or die by it.

**Exit ticket:** in one sentence, what is the one fact your **final project** agent must never be allowed to type?